# Подготовка набора данных

Подготовка библиотек и отключение предупреждений

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

# Импорт библиотеки
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

sns.set_theme(style="ticks")

Подключение Google Drive к блокноту Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Загрузка набора данных

Набор данных, используемый в лабораторных работах доступен по [ссылке](https://www.kaggle.com/datasets/mssmartypants/paris-housing-classification)

Поскольку работы выполнены в среде Google Colaboratory, путь до файла указывается относительно директории, к которой примонтирован Google диск

In [ ]:
ParisHousingClass_df = pd.read_csv('/content/drive/MyDrive/ParisHousingClass.csv')
ParisHousingClass_df.head()

,squareMeters,numberOfRooms,hasYard,hasPool,floors,cityCode,cityPartRange,numPrevOwners,made,isNewBuilt,hasStormProtector,basement,attic,garage,hasStorageRoom,hasGuestRoom,price,category
0,75523,3,0,1,63,9373,3,8,2005,0,1,4313,9005,956,0,7,7559081.5,Basic
1,80771,39,1,1,98,39381,8,6,2015,1,0,3653,2436,128,1,2,8085989.5,Luxury
2,55712,58,0,1,19,34457,6,8,2021,0,0,2937,8852,135,1,9,5574642.1,Basic
3,32316,47,0,0,6,27939,10,4,2012,0,1,659,7141,359,0,3,3232561.2,Basic
4,70429,19,1,1,90,38045,3,7,1990,1,0,8435,2429,292,1,4,7055052.0,Luxury


Разделение на матрицу признаков и зависимую переменную

In [ ]:
X = ParisHousingClass_df.drop(['price'], axis=1)
y = ParisHousingClass_df.price
print("Матрица признаков")
print(X.values)
print("Зависимая переменная")
print(y.values)

Матрица признаков
[[75523 3 0 ... 0 7 'Basic']
 [80771 39 1 ... 1 2 'Luxury']
 [55712 58 0 ... 1 9 'Basic']
 ...
 [83841 3 0 ... 1 9 'Basic']
 [59036 70 0 ... 1 4 'Basic']
 [1440 84 0 ... 1 6 'Basic']]
Зависимая переменная
[7559081.5 8085989.5 5574642.1 ... 8390030.5 5905107.   146708.4]


# Обработка категориальных данных

Замена категории кодом `LabelEncoder`

In [ ]:
from sklearn.preprocessing import LabelEncoder

category_encoder = LabelEncoder()
print("Категориальная переменная до обработки")
print(X.category)
encoded_category = category_encoder.fit_transform(X.category)
print("Категориальная переменная после обработки")
print(encoded_category)

Категориальная переменная до обработки
0        Basic
1       Luxury
2        Basic
3        Basic
4       Luxury
         ...  
9995     Basic
9996     Basic
9997     Basic
9998     Basic
9999     Basic
Name: category, Length: 10000, dtype: object
Категориальная переменная после обработки
[0 1 0 ... 0 0 0]


Метод `OneHotEncoder` используется для преобразования категориальных данных. В используемых данных отсутствуют категориальные переменные, у которых больше двух вариантов, в связи с чем метод не используется.

# Метод с применением класса ColumnTransformer

In [ ]:
# Создаем копию объекта: с пропусками и некодированными категориями
X_dirty = X.copy()
X_dirty

,squareMeters,numberOfRooms,hasYard,hasPool,floors,cityCode,cityPartRange,numPrevOwners,made,isNewBuilt,hasStormProtector,basement,attic,garage,hasStorageRoom,hasGuestRoom,category
0,75523,3,0,1,63,9373,3,8,2005,0,1,4313,9005,956,0,7,Basic
1,80771,39,1,1,98,39381,8,6,2015,1,0,3653,2436,128,1,2,Luxury
2,55712,58,0,1,19,34457,6,8,2021,0,0,2937,8852,135,1,9,Basic
3,32316,47,0,0,6,27939,10,4,2012,0,1,659,7141,359,0,3,Basic
4,70429,19,1,1,90,38045,3,7,1990,1,0,8435,2429,292,1,4,Luxury
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1726,89,0,1,5,73133,7,6,2009,0,1,9311,1698,218,0,4,Basic
9996,44403,29,1,1,12,34606,9,4,1990,0,1,9061,1742,230,0,0,Basic
9997,83841,3,0,0,69,80933,10,10,2005,1,1,8304,7730,345,1,9,Basic
9998,59036,70,0,0,96,55856,1,3,2010,0,1,2590,6174,339,1,4,Basic


Современный метод трансформации признаков

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer


categoricalColumns = ['category', 'made', 'numPrevOwners']
columnsToRescale = ['squareMeters', 'basement', 'attic', 'garage', 'floors']
droppedColumns = ['cityCode']
leaveColumns = X.columns.difference(categoricalColumns)
leaveColumns = leaveColumns.difference(columnsToRescale)
leaveColumns = leaveColumns.difference(droppedColumns)

# Создаем список трансформеров
transformers = [
    ('text_encoder', OrdinalEncoder(), categoricalColumns),
    ('data_scaler', StandardScaler(), columnsToRescale),
    ('preprocessed', 'passthrough', leaveColumns)
]
# Создаем объект ColumnTransformer и передаем ему список трансформеров
data_preprocessor = ColumnTransformer(transformers)
# Выполнем трансформацию признаков
X_transformed = data_preprocessor.fit_transform(X=X_dirty)
print(X_transformed.shape)
X_transformed

(10000, 16)


array([[ 0., 15.,  7., ...,  0.,  0.,  3.],
       [ 1., 25.,  5., ...,  1.,  1., 39.],
       [ 0., 31.,  7., ...,  0.,  0., 58.],
       ...,
       [ 0., 15.,  9., ...,  0.,  1.,  3.],
       [ 0., 20.,  2., ...,  0.,  0., 70.],
       [ 0.,  4.,  9., ...,  0.,  1., 84.]])

In [ ]:
columns_after_preprocessing = categoricalColumns.copy()
columns_after_preprocessing += columnsToRescale
columns_after_preprocessing += leaveColumns.to_list()
print(f'Ожидается колонок: {len(columns_after_preprocessing)}')
columns_after_preprocessing

Ожидается колонок: 16


['category',
 'made',
 'numPrevOwners',
 'squareMeters',
 'basement',
 'attic',
 'garage',
 'floors',
 'cityPartRange',
 'hasGuestRoom',
 'hasPool',
 'hasStorageRoom',
 'hasStormProtector',
 'hasYard',
 'isNewBuilt',
 'numberOfRooms']

Преобразование полученного многомерного массива обратно в `Dataframe`

In [ ]:
X_data = pd.DataFrame(
    X_transformed,
    columns=columns_after_preprocessing
)
X_data

,category,made,numPrevOwners,squareMeters,basement,attic,garage,floors,cityPartRange,hasGuestRoom,hasPool,hasStorageRoom,hasStormProtector,hasYard,isNewBuilt,numberOfRooms
0,0.0,15.0,7.0,0.891562,-0.250333,1.374130,1.537488,0.440453,3.0,7.0,1.0,0.0,1.0,0.0,0.0,3.0
1,1.0,25.0,5.0,1.073956,-0.479772,-0.895592,-1.622370,1.652041,8.0,2.0,1.0,1.0,0.0,1.0,1.0,39.0
2,0.0,31.0,7.0,0.203033,-0.728678,1.321265,-1.595657,-1.082685,6.0,9.0,1.0,1.0,0.0,0.0,0.0,58.0
3,0.0,22.0,3.0,-0.610092,-1.520589,0.730080,-0.740816,-1.532703,10.0,3.0,0.0,0.0,1.0,0.0,0.0,47.0
4,1.0,0.0,6.0,0.714521,1.182616,-0.898010,-0.996505,1.375106,3.0,4.0,1.0,1.0,0.0,1.0,1.0,19.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.0,19.0,5.0,-1.673244,1.487144,-1.150586,-1.278908,-1.567320,7.0,4.0,1.0,0.0,1.0,0.0,0.0,89.0
9996,0.0,0.0,3.0,-0.190009,1.400235,-1.135383,-1.233113,-1.325002,9.0,0.0,1.0,0.0,1.0,1.0,0.0,29.0
9997,0.0,15.0,9.0,1.180654,1.137076,0.933592,-0.794243,0.648154,10.0,9.0,0.0,1.0,1.0,0.0,1.0,3.0
9998,0.0,20.0,2.0,0.318559,-0.849307,0.395962,-0.817141,1.582807,1.0,4.0,0.0,1.0,1.0,0.0,0.0,70.0
